# Milestone 9: Roofline analysis

## 1. The T4's official peak numbers (and where they come from)

| Spec | Value | Source |
|---|---|---|
| FP32 (single-precision) peak | **8.1 TFLOP/s** = 8,100 GFLOP/s | NVIDIA T4 datasheet and product page (they agree) |
| Memory bandwidth | **"over 320 GB/s"** | NVIDIA T4 product page |
| Memory bandwidth | **300 GB/s** | NVIDIA T4 Tensor Core datasheet (PDF) |
| CUDA cores | 2,560 | both |
| Memory | 16 GB GDDR6 | both |

Sources:
- NVIDIA T4 product page: https://www.nvidia.com/en-gb/data-center/tesla-t4/
- NVIDIA T4 Tensor Core datasheet: https://nvidia.com/content/dam/en-zz/Solutions/Data-Center/tesla-t4/t4-tensor-core-datasheet.pdf

**NVIDIA's own two documents disagree on bandwidth (300 vs 320 GB/s).** We use 320 GB/s for the roofline, because a peak is a ceiling and the higher figure is the safer one. We **also measure** what a perfectly coalesced copy kernel actually achieves on your card. Real GPUs never reach the paper peak, so the measured number is the fairer target.

**Where 8.1 TFLOP/s comes from:** 2,560 cores × 2 operations per clock (one fused multiply-add = a multiply and an add) × about 1.59 GHz boost clock ≈ 8.1 trillion operations per second. It assumes *every* core does an FMA on *every* clock, which real code almost never does.

## 2. What a roofline is

Every kernel needs two resources: **memory traffic** (bytes) and **arithmetic** (operations). Its speed is limited by whichever runs out first.

- **Arithmetic intensity** = operations ÷ bytes. How much maths a kernel does for each byte it moves.
- **Memory roof:** at most `intensity × 320 GB/s` operations per second. That's a sloped line on a log-log chart.
- **Compute roof:** at most 8,100 GFLOP/s. That's a flat line.
- The two roofs meet at the **ridge point**: 8,100 ÷ 320 ≈ **25.3 FLOP/byte**.

A kernel **left of 25.3** is **memory-bound**: it can't go faster than memory delivers bytes, however fast the maths is. A kernel **right of 25.3** is **compute-bound**.

## 3. Counting by hand, per pixel

**Bytes** = the minimum that *must* cross global memory if every value is read once and written once. Real kernels can move more, if the cache misses; this is the unavoidable part.

**Operations:** add, subtract, multiply, divide and clamp each count 1, and an FMA counts 2. `powf` is counted as 1, even though it really takes about 10 instructions, so gamma is **under-counted**. Index arithmetic (integer maths for addresses) isn't counted at all.

| Stage | read | write | **bytes** | **operations** | how the operations add up | intensity |
|---|---|---|---|---|---|---|
| black level | 2 (uint16) | 4 (float) | **6** | **4** | subtract, divide, 2 clamps | 0.67 |
| demosaic | 4 | 12 | **16** | **12** | 4 averages: (3 adds + 1 divide) × 2 + (1 add + 1 divide) × 2 | 0.75 |
| white balance | 12 | 12 | **24** | **6** | 3 multiplies + 3 clamps | 0.25 |
| denoise | 12 | 12 | **24** | **57** | per colour: 9 FMAs (18) + 1 divide = 19, × 3 colours | 2.38 |
| gamma | 12 | 3 (bytes) | **15** | **15** | per colour: 2 clamps + powf + multiply + add = 5, × 3 | 1.00 |
| **5 kernels total** | | | **85** | **94** | | **1.11** |
| **fused** | 2 | 3 | **5** | **≈ 102.7** | see below | **≈ 20.5** |

**The fused kernel's operations:** it does everything above once per pixel, plus repeated work for the tile halos.
- black level runs on 36 × 12 = 432 values per 256 pixels: 4 × 432/256 = 6.75
- demosaic + white balance runs on 34 × 10 = 340 per 256: (12 + 6) × 340/256 = 23.9
- denoise + gamma: 57 + 15 = 72
- **total ≈ 102.7 operations for only 5 bytes**

**Prediction before measuring:** every separate stage has intensity between 0.25 and 2.4, all far left of the 25.3 ridge, so **all are memory-bound**. Fusion moves the pipeline from 1.1 to about 20.5 FLOP/byte, close to the ridge. So fusion changes the *kind* of limit, not just the speed.

## 4. Achieved numbers

From each measured time *t* (seconds) and the image's *N* pixels:
- **achieved bandwidth** = bytes per pixel × N ÷ t
- **achieved GFLOP/s** = operations per pixel × N ÷ t
- **% of roof** = achieved GFLOP/s ÷ (the lower of the two roofs at that intensity)

All times come from **one fresh run** (the `gpu_roofline` program), so they're directly comparable with each other.

**Runtime must be T4 GPU.**

## Step 1: Connect Drive and go to the project folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/isp_project

## Step 2: Write the new files
- `gpu_roofline_main.cu`: measures the copy-kernel bandwidth ceiling, each stage of the naive and planar pipelines, the shared-memory denoise, and the fused kernel. The only new kernel is `copyKernel`, the simplest possible memory-bound kernel (one coalesced read and one write per thread).
- `roofline_analysis.py`: holds the hand counts from the table above and computes everything from your measurements.

In [ ]:
%%writefile gpu_roofline_main.cu
// gpu_roofline_main.cu
// Milestone 9: collects the times the roofline analysis needs, all in ONE run
// so they are directly comparable:
//   1. the best memory bandwidth a simple copy kernel achieves on this GPU
//   2. per-stage times of the naive pipeline (interleaved, 16x16)
//   3. per-stage times of the planar pipeline (32x8)
//   4. the shared-memory denoise kernel on its own
//   5. the fused kernel
// Output lines look like "RESULT,<name>,<value>" for the Python script.
// Each measurement: 1 warm-up + 10 averaged runs (Milestone 4 method).

#include <iostream>
#include <vector>

#include "cuda_helpers.h"
#include "gpu_buffers.h"
#include "gpu_fused_kernel.h"
#include "gpu_kernels.h"
#include "gpu_planar_kernels.h"
#include "gpu_shared_denoise.h"
#include "isp_io.h"

const int TIMED_RUN_COUNT = 10;
const int STAGE_COUNT = 5;
const char* STAGE_NAMES[STAGE_COUNT] = {"black", "demosaic", "white", "denoise", "gamma"};

// The simplest possible memory-bound kernel: each thread copies one float.
// One read + one write per thread, perfectly coalesced, almost no maths.
// Its speed is the practical memory-bandwidth ceiling for our kernels.
__global__ void copyKernel(const float* source, float* destination, int valueCount) {
    int valueIndex = blockIdx.x * blockDim.x + threadIdx.x;
    if (valueIndex >= valueCount) {
        return;
    }
    destination[valueIndex] = source[valueIndex];
}

static dim3 makeGridShape(const ImageMetadata& metadata, int blockWidth, int blockHeight) {
    int blocksAcross = (metadata.imageWidth + blockWidth - 1) / blockWidth;
    int blocksDown = (metadata.imageHeight + blockHeight - 1) / blockHeight;
    dim3 gridShape(blocksAcross, blocksDown);
    return gridShape;
}

static float elapsedMilliseconds(cudaEvent_t startEvent, cudaEvent_t stopEvent) {
    float milliseconds = 0.0f;
    checkCudaError(cudaEventElapsedTime(&milliseconds, startEvent, stopEvent), "reading elapsed time");
    return milliseconds;
}

static void measureCopyBandwidth(DeviceBuffers& buffers, cudaEvent_t startEvent, cudaEvent_t stopEvent) {
    // Copy the big RGB float buffer (about 270 MB) into the other one.
    int valueCount = static_cast<int>(buffers.rgbFloatBytes / sizeof(float));
    int threadsPerBlock = 256;
    int blockCount = (valueCount + threadsPerBlock - 1) / threadsPerBlock;

    float totalMilliseconds = 0.0f;
    for (int runNumber = 0; runNumber <= TIMED_RUN_COUNT; runNumber++) {
        checkCudaError(cudaEventRecord(startEvent), "recording start event");
        copyKernel<<<blockCount, threadsPerBlock>>>(buffers.deviceRgb, buffers.deviceDenoised, valueCount);
        checkCudaError(cudaGetLastError(), "launching copyKernel");
        checkCudaError(cudaEventRecord(stopEvent), "recording stop event");
        checkCudaError(cudaEventSynchronize(stopEvent), "waiting for copyKernel");
        if (runNumber > 0) {
            totalMilliseconds = totalMilliseconds + elapsedMilliseconds(startEvent, stopEvent);
        }
    }
    float averageMilliseconds = totalMilliseconds / TIMED_RUN_COUNT;

    // Bytes moved = read the buffer once + write it once.
    double bytesMoved = 2.0 * static_cast<double>(buffers.rgbFloatBytes);
    double gigabytesPerSecond = bytesMoved / (averageMilliseconds / 1000.0) / 1.0e9;
    std::cout << "RESULT,copy_kernel_ms," << averageMilliseconds << std::endl;
    std::cout << "RESULT,copy_kernel_GBps," << gigabytesPerSecond << std::endl;
}

// Launches one stage of the naive (interleaved) or planar pipeline.
static void launchStage(int stage, bool usePlanarLayout, DeviceBuffers& buffers,
                        const ImageMetadata& metadata, dim3 gridShape, dim3 blockShape) {
    int imageWidth = metadata.imageWidth;
    int imageHeight = metadata.imageHeight;

    if (stage == 0) {
        blackLevelKernel<<<gridShape, blockShape>>>(buffers.deviceBayer, buffers.deviceNormalised,
                                                    imageWidth, imageHeight,
                                                    metadata.blackLevel, metadata.whiteLevel);
    } else if (stage == 1 && usePlanarLayout) {
        demosaicPlanarKernel<<<gridShape, blockShape>>>(buffers.deviceNormalised, buffers.deviceRgb,
                                                        imageWidth, imageHeight);
    } else if (stage == 1) {
        demosaicKernel<<<gridShape, blockShape>>>(buffers.deviceNormalised, buffers.deviceRgb,
                                                  imageWidth, imageHeight);
    } else if (stage == 2 && usePlanarLayout) {
        whiteBalancePlanarKernel<<<gridShape, blockShape>>>(buffers.deviceRgb, imageWidth, imageHeight,
                                                            metadata.whiteBalanceRed,
                                                            metadata.whiteBalanceGreen,
                                                            metadata.whiteBalanceBlue);
    } else if (stage == 2) {
        whiteBalanceKernel<<<gridShape, blockShape>>>(buffers.deviceRgb, imageWidth, imageHeight,
                                                      metadata.whiteBalanceRed,
                                                      metadata.whiteBalanceGreen,
                                                      metadata.whiteBalanceBlue);
    } else if (stage == 3 && usePlanarLayout) {
        denoisePlanarKernel<<<gridShape, blockShape>>>(buffers.deviceRgb, buffers.deviceDenoised,
                                                       imageWidth, imageHeight);
    } else if (stage == 3) {
        denoiseKernel<<<gridShape, blockShape>>>(buffers.deviceRgb, buffers.deviceDenoised,
                                                 imageWidth, imageHeight);
    } else if (usePlanarLayout) {
        gammaPlanarKernel<<<gridShape, blockShape>>>(buffers.deviceDenoised, buffers.deviceOutput,
                                                     imageWidth, imageHeight);
    } else {
        gammaKernel<<<gridShape, blockShape>>>(buffers.deviceDenoised, buffers.deviceOutput,
                                               imageWidth, imageHeight);
    }
    checkCudaError(cudaGetLastError(), "launching a pipeline stage");
}

// Times each of the 5 stages separately and prints RESULT,<prefix>_<stage>,<ms>.
static void measureStages(const char* prefix, bool usePlanarLayout, int blockWidth, int blockHeight,
                          DeviceBuffers& buffers, const ImageMetadata& metadata,
                          cudaEvent_t stageEvents[]) {
    dim3 blockShape(blockWidth, blockHeight);
    dim3 gridShape = makeGridShape(metadata, blockWidth, blockHeight);
    float summedMilliseconds[STAGE_COUNT] = {0.0f, 0.0f, 0.0f, 0.0f, 0.0f};

    for (int runNumber = 0; runNumber <= TIMED_RUN_COUNT; runNumber++) {
        checkCudaError(cudaEventRecord(stageEvents[0]), "recording event 0");
        for (int stage = 0; stage < STAGE_COUNT; stage++) {
            launchStage(stage, usePlanarLayout, buffers, metadata, gridShape, blockShape);
            checkCudaError(cudaEventRecord(stageEvents[stage + 1]), "recording stage event");
        }
        checkCudaError(cudaEventSynchronize(stageEvents[STAGE_COUNT]), "waiting for the pipeline");
        if (runNumber > 0) {
            for (int stage = 0; stage < STAGE_COUNT; stage++) {
                float stageMilliseconds = elapsedMilliseconds(stageEvents[stage], stageEvents[stage + 1]);
                summedMilliseconds[stage] = summedMilliseconds[stage] + stageMilliseconds;
            }
        }
    }
    for (int stage = 0; stage < STAGE_COUNT; stage++) {
        std::cout << "RESULT," << prefix << "_" << STAGE_NAMES[stage] << ","
                  << summedMilliseconds[stage] / TIMED_RUN_COUNT << std::endl;
    }
}

// Times the shared-memory denoise or the fused kernel (useFused picks which).
static void measureSingleKernel(bool useFused, DeviceBuffers& buffers, const ImageMetadata& metadata,
                                cudaEvent_t startEvent, cudaEvent_t stopEvent) {
    dim3 blockShape(32, 8);
    dim3 gridShape = makeGridShape(metadata, 32, 8);
    float totalMilliseconds = 0.0f;

    for (int runNumber = 0; runNumber <= TIMED_RUN_COUNT; runNumber++) {
        checkCudaError(cudaEventRecord(startEvent), "recording start event");
        if (useFused) {
            fusedPipelineKernel<<<gridShape, blockShape>>>(buffers.deviceBayer, buffers.deviceOutput,
                                                           metadata.imageWidth, metadata.imageHeight,
                                                           metadata.blackLevel, metadata.whiteLevel,
                                                           metadata.whiteBalanceRed,
                                                           metadata.whiteBalanceGreen,
                                                           metadata.whiteBalanceBlue);
        } else {
            denoiseSharedKernel<<<gridShape, blockShape>>>(buffers.deviceRgb, buffers.deviceDenoised,
                                                           metadata.imageWidth, metadata.imageHeight);
        }
        checkCudaError(cudaGetLastError(), "launching single kernel");
        checkCudaError(cudaEventRecord(stopEvent), "recording stop event");
        checkCudaError(cudaEventSynchronize(stopEvent), "waiting for single kernel");
        if (runNumber > 0) {
            totalMilliseconds = totalMilliseconds + elapsedMilliseconds(startEvent, stopEvent);
        }
    }
    if (useFused) {
        std::cout << "RESULT,fused_all," << totalMilliseconds / TIMED_RUN_COUNT << std::endl;
    } else {
        std::cout << "RESULT,shared_denoise," << totalMilliseconds / TIMED_RUN_COUNT << std::endl;
    }
}

int main() {
    ImageMetadata metadata;
    bool metadataOk = readMetadataFile("data/metadata.txt", metadata);
    if (!metadataOk) {
        return 1;
    }
    int pixelCount = metadata.imageWidth * metadata.imageHeight;
    std::vector<uint16_t> bayerPixels;
    bool bayerOk = readBayerFile("data/bayer.bin", pixelCount, bayerPixels);
    if (!bayerOk) {
        return 1;
    }
    std::cout << "RESULT,pixels," << pixelCount << std::endl;

    DeviceBuffers buffers;
    allocateDeviceBuffers(buffers, pixelCount);
    checkCudaError(cudaMemcpy(buffers.deviceBayer, bayerPixels.data(), buffers.bayerBytes,
                              cudaMemcpyHostToDevice), "copying Bayer data to the GPU");

    cudaEvent_t stageEvents[STAGE_COUNT + 1];
    for (int eventNumber = 0; eventNumber <= STAGE_COUNT; eventNumber++) {
        checkCudaError(cudaEventCreate(&stageEvents[eventNumber]), "creating event");
    }

    measureCopyBandwidth(buffers, stageEvents[0], stageEvents[1]);
    measureStages("naive", false, 16, 16, buffers, metadata, stageEvents);
    measureStages("planar", true, 32, 8, buffers, metadata, stageEvents);
    // The planar run left a valid planar RGB image in deviceRgb for this one.
    measureSingleKernel(false, buffers, metadata, stageEvents[0], stageEvents[1]);
    measureSingleKernel(true, buffers, metadata, stageEvents[0], stageEvents[1]);

    for (int eventNumber = 0; eventNumber <= STAGE_COUNT; eventNumber++) {
        checkCudaError(cudaEventDestroy(stageEvents[eventNumber]), "destroying event");
    }
    freeDeviceBuffers(buffers);
    return 0;
}


In [ ]:
%%writefile roofline_analysis.py
# roofline_analysis.py
# Milestone 9: roofline analysis.
# Runs /content/gpu_roofline (which prints the measured times), combines them
# with the hand-counted bytes and operations per pixel below, prints a table,
# and draws a roofline chart.

import subprocess
import matplotlib.pyplot as pyplot

# ---------------------------------------------------------------------------
# T4 peak numbers, from NVIDIA's official pages (see the notebook text):
#   FP32 peak: 8.1 TFLOPS = 8100 GFLOP/s (datasheet and product page agree)
#   Memory bandwidth: product page says "over 320 GB/s", datasheet says 300 GB/s.
#   We use 320 for the roofline and ALSO report the bandwidth we measure.
# ---------------------------------------------------------------------------
peakGigaflops = 8100.0
specBandwidthGigabytes = 320.0
ridgePoint = peakGigaflops / specBandwidthGigabytes   # FLOP per byte where the roofs meet

# ---------------------------------------------------------------------------
# Hand counts PER PIXEL. "Bytes" = the minimum that must cross global memory
# if every value is read once and written once (caches may cause extra traffic
# in practice; we count the unavoidable part). "Operations" = floating-point
# operations: add, subtract, multiply, divide, compare-and-clamp each count 1;
# a multiply-add (FMA) counts 2; powf is counted as 1 even though it is really
# about ten instructions (so gamma's count is an UNDER-estimate).
#
# Stage      read  write  operations    where the operations come from
# black        2     4        4         subtract, divide, clamp low, clamp high
# demosaic     4    12       12         4 averages: 3+1, 3+1, 1+1, 1+1 adds/divides
# white       12    12        6         3 multiplies + 3 clamps
# denoise     12    12       57         per colour 9 FMAs (18) + 1 divide = 19, x3
# gamma       12     3       15         per colour 2 clamps + powf + multiply + add = 5, x3
# 5 kernels   42    43       94         total: 85 bytes
#
# Fused kernel: reads only the raw uint16 (2 bytes) and writes 3 bytes = 5 bytes.
# But it repeats some work for the tile halos:
#   black level for 36x12 = 432 values per 256 pixels -> 4 x 432/256 = 6.75
#   demosaic + white balance for 34x10 = 340 per 256  -> 18 x 340/256 = 23.91
#   denoise 57 + gamma 15                              -> 72
#   total about 102.66 operations per pixel.
# ---------------------------------------------------------------------------
stageCounts = {
    'black':    {'bytes': 6.0,  'operations': 4.0},
    'demosaic': {'bytes': 16.0, 'operations': 12.0},
    'white':    {'bytes': 24.0, 'operations': 6.0},
    'denoise':  {'bytes': 24.0, 'operations': 57.0},
    'gamma':    {'bytes': 15.0, 'operations': 15.0},
}
fusedOperations = 4.0 * 432.0 / 256.0 + 18.0 * 340.0 / 256.0 + 57.0 + 15.0
fusedBytes = 5.0


def runMeasurements():
    completedRun = subprocess.run(['/content/gpu_roofline'], capture_output=True, text=True)
    if completedRun.returncode != 0:
        print('ERROR running gpu_roofline:')
        print(completedRun.stdout)
        print(completedRun.stderr)
    measurements = {}
    for line in completedRun.stdout.splitlines():
        parts = line.strip().split(',')
        if len(parts) == 3 and parts[0] == 'RESULT':
            measurements[parts[1]] = float(parts[2])
    return measurements


def analyseKernel(rowName, bytesPerPixel, operationsPerPixel, milliseconds, pixelCount, measuredBandwidth):
    seconds = milliseconds / 1000.0
    arithmeticIntensity = operationsPerPixel / bytesPerPixel
    achievedGigabytes = bytesPerPixel * pixelCount / seconds / 1.0e9
    achievedGigaflops = operationsPerPixel * pixelCount / seconds / 1.0e9

    # The roofline: the best a kernel with this intensity could possibly do
    # is the LOWER of the two roofs.
    memoryRoof = arithmeticIntensity * specBandwidthGigabytes
    attainableGigaflops = min(peakGigaflops, memoryRoof)
    if arithmeticIntensity < ridgePoint:
        predictedLimit = 'memory'
    else:
        predictedLimit = 'compute'

    analysis = {
        'name': rowName,
        'milliseconds': milliseconds,
        'intensity': arithmeticIntensity,
        'gigabytes': achievedGigabytes,
        'percentSpecBandwidth': 100.0 * achievedGigabytes / specBandwidthGigabytes,
        'percentMeasuredBandwidth': 100.0 * achievedGigabytes / measuredBandwidth,
        'gigaflops': achievedGigaflops,
        'percentPeakFlops': 100.0 * achievedGigaflops / peakGigaflops,
        'percentOfRoof': 100.0 * achievedGigaflops / attainableGigaflops,
        'predictedLimit': predictedLimit,
    }
    return analysis


def buildAnalyses(measurements):
    pixelCount = measurements['pixels']
    measuredBandwidth = measurements['copy_kernel_GBps']
    analyses = []

    for prefix in ['naive', 'planar']:
        totalMilliseconds = 0.0
        for stageName in ['black', 'demosaic', 'white', 'denoise', 'gamma']:
            stageMilliseconds = measurements[prefix + '_' + stageName]
            totalMilliseconds = totalMilliseconds + stageMilliseconds
            analyses.append(analyseKernel(prefix + ' ' + stageName,
                                          stageCounts[stageName]['bytes'],
                                          stageCounts[stageName]['operations'],
                                          stageMilliseconds, pixelCount, measuredBandwidth))
        analyses.append(analyseKernel(prefix + ' ALL 5', 85.0, 94.0,
                                      totalMilliseconds, pixelCount, measuredBandwidth))

    analyses.append(analyseKernel('shared denoise', stageCounts['denoise']['bytes'],
                                  stageCounts['denoise']['operations'],
                                  measurements['shared_denoise'], pixelCount, measuredBandwidth))
    analyses.append(analyseKernel('fused', fusedBytes, fusedOperations,
                                  measurements['fused_all'], pixelCount, measuredBandwidth))
    return analyses


def printReport(measurements, analyses):
    print('T4 peaks: FP32', peakGigaflops, 'GFLOP/s | memory', specBandwidthGigabytes,
          'GB/s (spec) | ridge point', round(ridgePoint, 1), 'FLOP/byte')
    print('Measured copy-kernel bandwidth: {:.1f} GB/s ({:.0f}% of the 320 GB/s spec)'.format(
        measurements['copy_kernel_GBps'], 100.0 * measurements['copy_kernel_GBps'] / specBandwidthGigabytes))
    print()
    header = '{:<16} {:>8} {:>9} {:>8} {:>9} {:>10} {:>9} {:>9} {:>9}'
    print(header.format('kernel', 'ms', 'FLOP/B', 'GB/s', '%spec BW', '%meas. BW',
                        'GFLOP/s', '%peak FL', 'roofline'))
    row = '{:<16} {:>8.3f} {:>9.2f} {:>8.1f} {:>8.0f}% {:>9.0f}% {:>9.1f} {:>8.1f}% {:>9}'
    for analysis in analyses:
        print(row.format(analysis['name'], analysis['milliseconds'], analysis['intensity'],
                         analysis['gigabytes'], analysis['percentSpecBandwidth'],
                         analysis['percentMeasuredBandwidth'], analysis['gigaflops'],
                         analysis['percentPeakFlops'], analysis['predictedLimit']))


def drawRoofline(measurements, analyses):
    figure, axes = pyplot.subplots(figsize=(9, 6))

    # The two roofs. Left of the ridge, the sloped memory roof is lower;
    # right of it, the flat compute roof is lower.
    intensityValues = []
    intensity = 0.05
    while intensity <= 200.0:
        intensityValues.append(intensity)
        intensity = intensity * 1.1
    roofValues = []
    measuredRoofValues = []
    for intensity in intensityValues:
        roofValues.append(min(peakGigaflops, intensity * specBandwidthGigabytes))
        measuredRoofValues.append(min(peakGigaflops, intensity * measurements['copy_kernel_GBps']))
    axes.plot(intensityValues, roofValues, color='#57564f', linewidth=2,
              label='T4 roof (320 GB/s spec, 8.1 TFLOP/s)')
    axes.plot(intensityValues, measuredRoofValues, color='#57564f', linewidth=1.5, linestyle='--',
              label='Measured copy bandwidth roof')

    # One colour and marker per version (validated categorical palette, Milestone 8).
    groupStyles = {
        'naive':  {'colour': '#eb6834', 'marker': 's', 'label': 'naive stages (M2)'},
        'planar': {'colour': '#1baf7a', 'marker': '^', 'label': 'planar stages (M5)'},
        'shared': {'colour': '#eda100', 'marker': 'D', 'label': 'shared denoise (M6)'},
        'fused':  {'colour': '#e87ba4', 'marker': 'v', 'label': 'fused (M7)'},
    }
    labelledGroups = []
    for analysis in analyses:
        groupName = analysis['name'].split(' ')[0]
        style = groupStyles[groupName]
        legendLabel = None
        if groupName not in labelledGroups:
            legendLabel = style['label']
            labelledGroups.append(groupName)
        axes.scatter(analysis['intensity'], analysis['gigaflops'], color=style['colour'],
                     marker=style['marker'], s=70, edgecolors='white', linewidths=1.5,
                     zorder=3, label=legendLabel)
        # Label only the planar stages and the fused kernel, to avoid clutter.
        if groupName == 'planar' or groupName == 'fused':
            axes.annotate(analysis['name'], (analysis['intensity'], analysis['gigaflops']),
                          textcoords='offset points', xytext=(7, -3), fontsize=8, color='#57564f')

    axes.set_xscale('log')
    axes.set_yscale('log')
    axes.set_xlabel('Arithmetic intensity (FLOP per byte of global memory traffic)')
    axes.set_ylabel('Achieved performance (GFLOP/s)')
    axes.set_title('Roofline: T4, full image (' + str(int(measurements['pixels'])) + ' pixels)')
    axes.grid(True, which='major', color='#e4e3df', linewidth=0.8)
    axes.spines['top'].set_visible(False)
    axes.spines['right'].set_visible(False)
    axes.legend(frameon=False, fontsize=9, loc='lower right')
    figure.tight_layout()
    figure.savefig('output/roofline_chart.png', dpi=130)
    print()
    print('Saved output/roofline_chart.png')


def main():
    measurements = runMeasurements()
    analyses = buildAnalyses(measurements)
    printReport(measurements, analyses)
    drawRoofline(measurements, analyses)


main()


## Step 3: Compile
**Expected:** no output.

In [ ]:
!nvcc -O2 -arch=sm_75 -o /content/gpu_roofline gpu_roofline_main.cu gpu_fused_kernel.cu gpu_shared_denoise.cu gpu_planar_kernels.cu gpu_buffers.cu gpu_pointwise_kernels.cu gpu_neighbourhood_kernels.cu cuda_helpers.cu isp_io.cpp

## Step 4: Run the analysis (CHECKPOINT 1)
**Expected:** the T4 peaks line, the measured copy-kernel bandwidth, then a table with one row per kernel: time, intensity (FLOP/B), achieved GB/s, % of spec and of measured bandwidth, achieved GFLOP/s, % of peak FLOP/s, and the roofline's prediction.

**Paste the whole output to me.** I'll write up which kernels are limited by what, using your numbers.

In [ ]:
!python roofline_analysis.py

## Step 5: View the roofline chart (CHECKPOINT 2)
The solid line is the official roof and the dashed line is the bandwidth your GPU actually achieved. A point *on* the sloped roof means the kernel uses all the memory bandwidth there is. A point far *below* both roofs means something else is holding it back.

Just tell me it displayed.

In [ ]:
from IPython.display import Image as NotebookImage
from IPython.display import display
display(NotebookImage('output/roofline_chart.png'))